# Lecture 7 — Linear Combination of Unitaries (UnitaryLab companion)

Given unitaries $U_1,\dots,U_m$ and coefficients $c_j > 0$, the **LCU** primitive
implements the (generally non-unitary) operator $\sum_j c_j U_j$ on a target
state. A "prepare–select–unprepare" circuit applies $\sum_j c_j U_j$ to
$|\psi\rangle$ conditioned on measuring all ancillas in $|0\rangle$; that happens
with probability

$$p_{\text{succ}} \;=\; \frac{\big\|\sum_j c_j U_j |\psi\rangle\big\|^2}{\|c\|_1^2},
\qquad \|c\|_1 = \sum_j c_j.$$

The $\|c\|_1$ in the denominator is the LCU analogue of the block-encoding
subnormalization $\alpha$. This notebook uses `ua.LCUAlgorithm` to realize
$c_1 X + c_2 I$ and $c_1 X + c_2 Z$, and verifies each against the classical
matrix–vector product.

Contents:
1. Setup
2. Introspect the LCU result dictionary
3. $c_1 X + c_2 I$ on $|0\rangle$
4. $c_1 X + c_2 Z$ on a general state
5. The $\|c\|_1$ success probability
6. Your turn

In [1]:
import numpy as np
import unitarylab_algorithms as ua
from unitarylab.core import Circuit
import contextlib, io

# Suppress the algorithms' verbose logging so cell outputs stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield

np.set_printoptions(precision=4, suppress=True)
print("ready: unitarylab_algorithms loaded")

ready: unitarylab_algorithms loaded


## 2. Introspect the LCU result dictionary

`ua.LCUAlgorithm().run(alphas, unitaries, n_sys, initial_state=None, backend=...)`
takes the coefficients, a list of `Circuit` unitaries, and the number of system
qubits. Before asserting anything we print the returned dictionary's keys and
inspect the numeric fields.

In [2]:
u_x = Circuit(1); u_x.x(0)      # U1 = X
u_i = Circuit(1)                # U2 = I (empty circuit)

with quiet():
    res = ua.LCUAlgorithm().run([0.6, 0.4], [u_x, u_i], 1, backend="numpy")

print("keys:", list(res.keys()))
print("Result state       :", np.round(res["Result state"], 4))
print("Success probability :", round(res["Success probability"], 4))

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Success probability', 'Computation time (s)', 'Result state']
Result state       : [0.4+0.j 0.6+0.j]
Success probability : 0.52


## 3. $c_1 X + c_2 I$ on $|0\rangle$

With $c = (0.6, 0.4)$ acting on $|0\rangle$, the classical target is
$0.6\,X|0\rangle + 0.4\,I|0\rangle = 0.6|1\rangle + 0.4|0\rangle$. The algorithm
returns the *unnormalized* system state (its norm carries the success-probability
information), so we compare directions after normalizing both.

In [3]:
def normalize(v):
    return v / np.linalg.norm(v)

I2 = np.eye(2); X = np.array([[0, 1], [1, 0]], dtype=complex)
psi0 = np.array([1, 0], dtype=complex)
c1, c2 = 0.6, 0.4

target = c1 * X @ psi0 + c2 * I2 @ psi0
got = res["Result state"]
print("LCU result (raw)  :", np.round(got, 4), " norm =", round(np.linalg.norm(got), 4))
print("classical target  :", np.round(target, 4))
print("normalized LCU    :", np.round(normalize(got), 4))
print("normalized target :", np.round(normalize(target), 4))

# Numpy statevector reproduced exactly up to global normalization; tolerance 1e-9.
assert np.allclose(normalize(got), normalize(target), atol=1e-9)
print("\nVerified: LCU output direction matches (c1 X + c2 I)|0>.")

LCU result (raw)  : [0.4+0.j 0.6+0.j]  norm = 0.7211
classical target  : [0.4+0.j 0.6+0.j]
normalized LCU    : [0.5547+0.j 0.8321+0.j]
normalized target : [0.5547+0.j 0.8321+0.j]

Verified: LCU output direction matches (c1 X + c2 I)|0>.


## 4. $c_1 X + c_2 Z$ on a general state

A second example with a non-trivial input state and two non-commuting Paulis.
We prepare $|\psi\rangle = R_y(\theta)|0\rangle$ as the LCU `initial_state` and
check against $c_1 X|\psi\rangle + c_2 Z|\psi\rangle$.

In [4]:
Z = np.array([[1, 0], [0, -1]], dtype=complex)
theta = 0.9
psi = np.array([np.cos(theta / 2), np.sin(theta / 2)], dtype=complex)

u_x = Circuit(1); u_x.x(0)
u_z = Circuit(1); u_z.z(0)
init = Circuit(1); init.ry(theta, 0)     # prepares R_y(theta)|0> = psi
c1, c2 = 0.7, 0.5

with quiet():
    res2 = ua.LCUAlgorithm().run([c1, c2], [u_x, u_z], 1, initial_state=init, backend="numpy")

target2 = c1 * X @ psi + c2 * Z @ psi
got2 = res2["Result state"]
print("normalized LCU    :", np.round(normalize(got2), 4))
print("normalized target :", np.round(normalize(target2), 4))
assert np.allclose(normalize(got2), normalize(target2), atol=1e-9)
print("Verified: (c1 X + c2 Z)|psi> reproduced (direction, tol 1e-9).")

normalized LCU    : [0.8773+0.j 0.4799+0.j]
normalized target : [0.8773+0.j 0.4799+0.j]
Verified: (c1 X + c2 Z)|psi> reproduced (direction, tol 1e-9).


## 5. The $\|c\|_1$ success probability

The heralded success probability is
$p_{\text{succ}} = \|\sum_j c_j U_j|\psi\rangle\|^2 / \|c\|_1^2$. We reconstruct it
classically and compare to the value the algorithm reports. This makes the cost of
a large $\|c\|_1$ concrete: doubling $\|c\|_1$ quarters the success probability.

In [5]:
c = np.array([c1, c2])
l1 = c.sum()
p_classical = np.linalg.norm(target2) ** 2 / l1 ** 2
print("||c||_1              =", round(l1, 4))
print("p_succ (classical)   =", round(p_classical, 4))
print("p_succ (algorithm)   =", round(res2["Success probability"], 4))
assert abs(p_classical - res2["Success probability"]) < 1e-6
print("Verified: success probability = ||sum c_j U_j |psi>||^2 / ||c||_1^2.")

||c||_1              = 1.2
p_succ (classical)   = 0.5139
p_succ (algorithm)   = 0.5139
Verified: success probability = ||sum c_j U_j |psi>||^2 / ||c||_1^2.


## 6. Your turn
- Realize $\tfrac12(I + Z) = |0\rangle\!\langle 0|$ as an LCU and confirm it projects.
- Build a 3-term LCU $c_1 I + c_2 X + c_3 Z$ and verify against numpy.
- Vary the coefficients toward $\|c\|_1 \gg 1$ and tabulate the shrinking
  success probability — the LCU cost model in action.

See Lecture 7 for how LCU composes block encodings (sums and products) and feeds
into qubitization.